Description: 
given the target configuration in joint space, finding the movement trajectory.


In [1]:
import torch, time
import sys, os
sys.path.append('./../')

import numpy as np
from panda_robot.config import absjoin, PANDA_XML, SHELF_URDF, BASE_URDF, \
    BASE_URDF

import sys, os


from envs.env_panda_shelf_gs import PandaShelf

from src.ttts_augment import TTTS
from robot_utils import Point2PointMotion, PlanarManipulatorCost


%load_ext autoreload
%autoreload 2

device = 'cuda' if torch.cuda.is_available() else 'cpu'

[I 07/03/25 11:30:57.170 191976] [shell.py:_shell_pop_print@23] Graphical python shell detected, using wrapped sys.stdout


In [2]:
ARM_BASE_HEIGHT = 0.7

MAX_BATCH = 1000

In [3]:
def env_infos_gen():
    """
    This function generates the environment information 
    useful for defining the simulation environment.
    """
    shelf_info = {
        "name": "shelf",
        "urdf": SHELF_URDF,
        "base_pose": [0.7, 0.0, 0.015, 0, 0, 90],
    }
    print("SHELF_URDF: ", SHELF_URDF)

    base_info = {
        "name": "base",
        "urdf": BASE_URDF,
        "base_pose": [0.0, 0.0, 0.35, 0, 0, 90]
    }

    robot_info = {
        "name": "panda",
        "xml": PANDA_XML,
        "base_pose": [0.0, 0.0, 0.8, 0, 0, 0],
        # "conf": [0.0, -np.pi/4, 0.0, -np.pi/2, 
        #          0, np.pi/2, np.pi/4, 0.06, 0.06],
        "conf": [1.76, -4.5e-01, -1.77, -1.35,  4.9e-01,
                2.36, -2.8e-01, 0.06, 0.06],
        "jnt_names": ["joint1", "joint2", "joint3",
                    "joint4", "joint5", "joint6",
                    "joint7", "finger_joint1", "finger_joint2", ],
        "kp": np.array([4500, 4500, 3500, 3500, 2000, 2000, 
                            2000, 100, 100]),
        "kv": np.array([450, 450, 350, 350, 200, 200, 200, 
                        10, 10]),
        "force_range": {
            "lower": np.array([-87, -87, -87, -87, -12, -12, 
                                -12, -100, -100]),
            "upper": np.array([87,  87,  87, 87,  12,  12, 
                                12,  100,  100]),
        },
    }

    goal_info = {
        "goal_config": [ 4.3105, -0.7813, -4.4040, -2.4188, -0.5002,  4.6439, -1.9318,  0.0000,
          0.0000]
        # [2.6170, -2.1647, -1.3917, -2.0655,  2.4773,  0.7699, -1.7667,  0.0000,
        #   0.0000]
    }

    

    utilities = {
        "target_x1": [0.70, -0.15, 1.5],
        "target_x2": [0.60, 0.15, 1.1],
        # "1": [0.60, 0.15, 1.5],
        # "2": [0.55, -0.2, 1.6],
        # "3": [0.5, -0.2, 1.2],
        # "4": [0.6, 0.15, 1.1],
        # "5": [0.7, 0.1, 1.1],
        # "6": [0.6, -0.1, 1.1],
    }

    env_infos = {
        "shelf": shelf_info,
        "base": base_info,
        "robot": robot_info,
        "utilities": utilities,
        "goal_info": goal_info,
    }

    return env_infos

def test_collision(env_infos):
    action1 = [[0.0, -np.pi/4, 0.0, -np.pi/2, 
                 0, np.pi/2, np.pi/4, 0.06, 0.06]]*2
    action2 = [[0.0, -np.pi/4, 0.0, -np.pi/2, 
                 0, np.pi/2, np.pi/4, 0.06, 0.06]]*8
    
    action = action1 + action2
    action = torch.tensor(action, dtype=torch.float32)

    return action



In [4]:
# Define CMA-ES parameters
num_guess = 5
popsize = 100
num_iterations = 10
# n_envs = num_guess * popsize
max_batch = MAX_BATCH
K = 2

# Define p2p motion
T = 1
dt = 0.01
n_joints = 7
basis = 'bs'
bounds = None
p2p_motion = Point2PointMotion(T=T, dt=dt, K=K, basis=basis, n=n_joints,
                            bounds=bounds, device=device)


In [5]:
# Generate the environment information
env_infos = env_infos_gen()

In [6]:
env = PandaShelf(n_envs=max_batch, gpu=True, use_gui=False,
                p2p_motion=p2p_motion)
env.load_world(env_infos)
# Define the initial_guesses

task_goal = env_infos["goal_info"]["goal_config"]

# x_goal = torch.tensor(task_goal).to(device)

# initial robot configuration
# init for target x1: 
init1 = [-0.063, 0.159, -0.152, -1.260, -0.064, 2.274, 0.785]
# initi for target x2:
init2 = [0.199, 0.146, 0.023, -1.703, 0.082, 1.900, 0.793]
# initial_sol = torch.tile(torch.tensor(init2, device=device, dtype=torch.float32),
#                         (num_guess, 1))
# initial_w = torch.rand((num_guess, K*n_joints), device=device)
# initial_guesses = torch.cat((initial_sol, initial_w), dim=-1).to(device)

x_min = torch.tensor([0.45, -0.3, 0.8]).to(device)
x_max = torch.tensor([0.75, 0.3, 1.7]).to(device)

dh_x = 20

domain_task = [torch.linspace(x_min[i], x_max[i], 20) for i in range(3)]

# parallel CMAES optimization
# TODO domain is not specified correctly here
max_theta = torch.pi
min_theta = -1*max_theta
d0_theta = 20; 
domain_theta = [torch.linspace(min_theta, max_theta, d0_theta).to(device)]*n_joints
domain_w_theta = [torch.linspace(min_theta, max_theta, d0_theta).to(device)]*(K*n_joints)
# domain = domain_theta + domain_w_theta
domain = domain_w_theta

[Genesis] [11:31:05] [WARNING] Overriding base link's pos with user provided value in morph.
[Genesis] [11:31:05] [WARNING] Overriding base link's quat with user provided value in morph.


In [9]:

def objective_func(xs):
    global evaluate_time
    st_time = time.time()
    """
    This function calculates the costs given the trajectories.
    Args:
        x: (num_guess*popsize, n_joints + K*n_joints) tensor
    Returns:
        costs: (num_guess*popsize,) tensor
    """
    xs_pre = xs.clone()
    if xs.shape[0] < MAX_BATCH:
        xs = torch.cat([xs, torch.zeros((MAX_BATCH - xs.shape[0], xs.shape[1]), device=xs.device)], dim=0)

    # x_goal = xs[:, :3]
    # x = xs[:, 3:]
    x = xs

    env.reset(env_infos)
    robot = env_infos["robot"]["id"]
    theta_0 = robot.get_qpos()[:, :n_joints]

    """cost_collision"""
    # generate trajectories given the wights
    theta_t = env.get_traj_p2(x, theta_0) #batchxtimexjoint_angle
    # propage the joint configurations
    collisions = env.forward_configuration(env_infos, theta_t)
    # sum the collision flag at each time step as collision cost
    d_obst = torch.sum(collisions, dim=-1)

    """cost control"""
    d_control = env.dist_traj(theta_t)


    """cost_goal"""
    task_goal = env_infos["goal_info"]["goal_config"]
    task_goal = torch.tensor(task_goal).to(device)
    d_goal = torch.linalg.norm(theta_t[:, -1, :] - task_goal[:7], dim=-1)
    # read robot's eef position
    # eef = robot.get_link("grasping_target_hand")
    # eef_pos = eef.get_pos()
    # # print("eef position", eef_pos)
    # d_goal = torch.linalg.norm(eef_pos - x_goal, dim=-1)
    # d_goal = torch.linalg.norm(theta_t[:, -1, :] - target_q, dim=-1)
    d_total = 50*d_goal + 1*d_obst + 0.1*d_control

    if xs_pre.shape[0] < MAX_BATCH:
        d_total = d_total[:xs_pre.shape[0]]

    evaluate_time += time.time() - st_time

    return torch.exp(-(d_total))

In [10]:
model_folder = os.getcwd() + "/tt_models/"
file_name = 'mp_7dof_test_config'
save_path = os.path.join(model_folder, file_name)
evaluate_time = 0
ttts = TTTS(func = objective_func, domain = domain, value_k=5, visit_k=5, 
            cross_max_iter=2, kickrank=20, max_tt=100, max_batch=MAX_BATCH,
        warm_start=True, verbose=False, save_path = save_path)#, test_mode=True)


The initial tt model is loaded!


In [11]:
# ttts.task_var = torch.tensor(task_goal)[None, :].to(device)
ttts.task_var = None

sol, mcts_time, tt_x = ttts.iterate(max_iters =1, param_C=3, test_mode=True) # K x n
sol_2, cmaes_time = ttts.parallel_cmaes_gpu(sol, num_iterations=50, test_mode=True, sigma=0.25)

mcts-iters: 1, top-k values: [9.939181976100246e-09, 9.904886467190292e-09, 9.825683700899605e-09, 9.819506095063916e-09, 9.784991831166424e-09]


In [12]:
print("sol_2: ", sol_2.shape)

sol_2:  torch.Size([5, 14])


In [13]:
env.reset(env_infos)
robot = env_infos["robot"]["id"]
shelf = env_infos["shelf"]["id"]
theta_0 = robot.get_qpos()[:num_guess, :n_joints]
theta_t = env.get_traj_p2(sol_2, theta_0)

# save the trajectory
np.save("mp_traj_configuration.npy", theta_t.cpu().numpy())

In [14]:
theta_t.shape

torch.Size([5, 108, 7])